In [ ]:
import os
from google.colab import drive, auth
import zipfile

# Google Drive 마운트 및 API 인증
drive.mount('/content/drive')
auth.authenticate_user() # Google Drive API 사용을 위한 인증

# 저장할 구글 드라이브 폴더 ID (사용자가 제공한 URL에서 추출)
drive_folder_id = '1teiRbwW4ciSrKLd4bvs7kc81Y5q3lkwV'

# 로컬 저장 경로 설정
output_dir = '/content/output_files'
os.makedirs(output_dir, exist_ok=True)

## 데이터 불러오기

In [ ]:
#통합된 데이터 사용

import pandas as pd
df = pd.read_csv('/content/final_data.csv')
print(df.head())

                                      merged_content
0  우리 동네에 하이마트랑 엘지가전 또 무슨 가전판매건물 세 개가 뭉쳐있는데 장사가 잘...
1  어디서 사든 어차피 같은 창고서 꺼내주는데; 모델명 뒷자리까지 동일한지 체크나 잘 ...
2  오프라인에서 10만원 정도 화장품을 온라인에선 5~7만원 정도면 구매할 수 있는데 ...
3  이제 결혼을 앞두고 있고 가전을 사려 하고 있어요. 어디에서 사는게 가장 경제적일까...
4  요즘 가전가격 보고 있는데 인터넷이랑 오프라인 차이가 너무 큼 매장에서 세트로사면 ...


In [ ]:
import re

def clean_text(text):

    # URL 제거
    text = re.sub(r"http\S+", "", text)

    # HTML 제거
    text = re.sub(r"<.*?>", "", text)

    # 멘션 제거
    text = re.sub(r"@\w+", "", text)

    # ㅋㅋㅋㅋ → ㅋㅋ
    text = re.sub(r"(ㅋ)\1{2,}", "ㅋㅋ", text)

    # ㅎㅎㅎㅎ → ㅎㅎ
    text = re.sub(r"(ㅎ)\1{2,}", "ㅎㅎ", text)

    # !!!!! → !
    text = re.sub(r"!{2,}", "!", text)

    # ????? → ?
    text = re.sub(r"\?{2,}", "?", text)

    # 공백 정리
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"중복값 제거 후 {initial_rows - len(df)}개의 행이 삭제되었습니다. 남은 행 수: {len(df)}")
df['merged_content'] = df['merged_content'].apply(clean_text)

중복값 제거 후 21475개의 행이 삭제되었습니다. 남은 행 수: 40546


## 모델 불러오기

In [ ]:
import torch
import warnings
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

device = "cuda" # Changed to CPU as requested
print(f"Using device: {device}")

model = SentenceTransformer(
    "nlpai-lab/KURE-v1",#"nomic-ai/nomic-embed-text-v1",
    trust_remote_code=True,
    device=device
)

# clustering 용도라면 prefix 붙이기
texts = df["merged_content"].astype(str).apply(lambda x: f"clustering: {x}").tolist()

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    batch_size=8, # Reduced batch size for CPU to avoid memory issues
    show_progress_bar=True
)

df["vector"] = embeddings.tolist()
df.head()

Using device: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/5069 [00:00<?, ?it/s]

,merged_content,vector
0,우리 동네에 하이마트랑 엘지가전 또 무슨 가전판매건물 세 개가 뭉쳐있는데 장사가 잘...,"[-0.058408938348293304, 0.013541605323553085, ..."
1,어디서 사든 어차피 같은 창고서 꺼내주는데; 모델명 뒷자리까지 동일한지 체크나 잘 ...,"[-0.044038865715265274, -0.02158508263528347, ..."
2,오프라인에서 10만원 정도 화장품을 온라인에선 5~7만원 정도면 구매할 수 있는데 ...,"[-0.03830742463469505, -0.01926391012966633, -..."
3,이제 결혼을 앞두고 있고 가전을 사려 하고 있어요. 어디에서 사는게 가장 경제적일까...,"[-0.006432096473872662, 0.038760218769311905, ..."
4,요즘 가전가격 보고 있는데 인터넷이랑 오프라인 차이가 너무 큼 매장에서 세트로사면 ...,"[-0.07503452897071838, -0.014947115443646908, ..."


In [ ]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import os

# DataFrame 저장 (CSV)
df_filename = os.path.join(output_dir, 'kure_embedding_data.csv')
df.to_csv(df_filename, index=False)
print(f"DataFrame saved to {df_filename}")

# Google Drive API를 사용하여 CSV 파일 업로드
drive_service = build('drive', 'v3')

if os.path.exists(df_filename):
    file_metadata = {
        'name': os.path.basename(df_filename),
        'parents': [drive_folder_id]
    }
    media = MediaFileUpload(df_filename, resumable=True)
    uploaded_file = drive_service.files().create(
        body=file_metadata,
        media_body=media,
        fields='id'
    ).execute()
    print(f"Uploaded '{os.path.basename(df_filename)}' to Google Drive folder ID: {drive_folder_id}. File ID: {uploaded_file.get('id')}")
else:
    print(f"Warning: CSV file not found for upload - {df_filename}")

print("CSV file has been processed and uploaded to Google Drive.")

## 클러스터링 시각화해서 확인하기

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from matplotlib import pyplot as plt

## 덴드로그램

In [ ]:
import os

link_model = linkage(list(df['vector']), 'ward') #ward 병합 기준으로

# Individual plot is no longer saved separately
plt.figure(figsize=(15,9))
dendrogram(link_model,
           orientation='top',
           distance_sort='descending',
           show_leaf_counts=False)
plt.title('Dendrogram of Kure Embeddings')

output_dir = '/content/output_files'
os.makedirs(output_dir, exist_ok=True)
dendrogram_filename = os.path.join(output_dir, 'dendrogram.png')
plt.savefig(dendrogram_filename)
plt.show()
print(f"Dendrogram saved to {dendrogram_filename}")
plt.close() # Close the plot to free memory

## UMAP

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
umap_reducer = umap.UMAP(n_components=2, random_state=42)

embedding = umap_reducer.fit_transform(list(df['vector']))

df['umap_x'] = embedding[:, 0]
df['umap_y'] = embedding[:, 1]

# Individual plot is no longer saved separately
plt.figure(figsize=(12, 10))
sns.scatterplot(
    x='umap_x',
    y='umap_y',
    data=df,
    s=10, # 점 크기
    alpha=0.7 # 투명도
)
plt.title('UMAP Projection of Kure Embeddings', fontsize=16)
plt.xlabel('UMAP Dimension 1', fontsize=12)
plt.ylabel('UMAP Dimension 2', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

output_dir = '/content/output_files'
os.makedirs(output_dir, exist_ok=True)
umap_filename = os.path.join(output_dir, 'umap_projection.png')
plt.savefig(umap_filename)
plt.show()
print(f"UMAP plot saved to {umap_filename}")
plt.close() # Close the plot to free memory

## PCA

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import os

# PCA 모델 설정
# n_components는 시각화를 위해 2로 설정합니다.
pca = PCA(n_components=2, random_state=42)

# Ko-SBERT 벡터에 PCA 적용
# result_df['vector']는 numpy array 형태로 변환되어야 합니다.
pca_embedding = pca.fit_transform(list(df['vector']))

# 차원 축소된 데이터를 DataFrame에 추가
df['pca_x'] = pca_embedding[:, 0]
df['pca_y'] = pca_embedding[:, 1]

# Individual plot is no longer saved separately
plt.figure(figsize=(12, 10))
sns.scatterplot(
    x='pca_x',
    y='pca_y',
    data=df,
    s=10, # 점 크기
    alpha=0.7 # 투명도
)
plt.title('PCA Projection of Kure Embeddings', fontsize=16)
plt.xlabel('PCA Dimension 1', fontsize=12)
plt.ylabel('PCA Dimension 2', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

output_dir = '/content/output_files'
os.makedirs(output_dir, exist_ok=True)
pca_filename = os.path.join(output_dir, 'pca_projection.png')
plt.savefig(pca_filename)
plt.show()
print(f"PCA plot saved to {pca_filename}")
plt.close() # Close the plot to free memory

## t-SNE

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # numpy import 추가
import os

# t-SNE 모델 설정
# n_components는 시각화를 위해 2로 설정합니다.
# perplexity, learning_rate, n_iter 등의 파라미터는 데이터에 따라 조정이 필요할 수 있습니다.
# random_state를 설정하여 결과의 재현성을 확보합니다.
tsne_model = TSNE(n_components=2, random_state=42, perplexity=30, learning_rate=200, n_iter=1000)

# Doc2Vec 벡터에 t-SNE 적용
# result_df['vector']는 numpy array 형태로 변환되어야 합니다.
# list(result_df['vector'])를 numpy 배열로 변환합니다.
tsne_embedding = tsne_model.fit_transform(np.array(list(df['vector'])))

# 차원 축소된 데이터를 DataFrame에 추가
df['tsne_x'] = tsne_embedding[:, 0]
df['tsne_y'] = tsne_embedding[:, 1]

# Individual plot is no longer saved separately
plt.figure(figsize=(12, 10))
sns.scatterplot(
    x='tsne_x',
    y='tsne_y',
    data=df,
    s=10, # 점 크기
    alpha=0.7 # 투명도
)
plt.title('t-SNE Projection of Kure Embeddings', fontsize=16)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

output_dir = '/content/output_files'
os.makedirs(output_dir, exist_ok=True)
tsne_filename = os.path.join(output_dir, 'tsne_projection.png')
plt.savefig(tsne_filename)
plt.show()
print(f"t-SNE plot saved to {tsne_filename}")
plt.close() # Close the plot to free memory

## 파일 공유 드라이브에 저장

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import zipfile

os.makedirs(output_dir, exist_ok=True)

# 1. 데이터프레임 저장 (CSV)
df_filename = os.path.join(output_dir, 'processed_data.csv')
df.to_csv(df_filename, index=False)
print(f"DataFrame saved to {df_filename}")

# 2. 모든 결과 파일을 하나의 ZIP 파일로 압축
zip_filename = os.path.join(output_dir, 'Nomic.zip')
with zipfile.ZipFile(zip_filename, 'w') as zf:
    if os.path.exists(df_filename):
        zf.write(df_filename, os.path.basename(df_filename))
    if os.path.exists(os.path.join(output_dir, 'dendrogram.png')):
        zf.write(os.path.join(output_dir, 'dendrogram.png'), os.path.basename(os.path.join(output_dir, 'dendrogram.png')))
    if os.path.exists(os.path.join(output_dir, 'umap_projection.png')):
        zf.write(os.path.join(output_dir, 'umap_projection.png'), os.path.basename(os.path.join(output_dir, 'umap_projection.png')))
    if os.path.exists(os.path.join(output_dir, 'pca_projection.png')):
        zf.write(os.path.join(output_dir, 'pca_projection.png'), os.path.basename(os.path.join(output_dir, 'pca_projection.png')))
    if os.path.exists(os.path.join(output_dir, 'tsne_projection.png')):
        zf.write(os.path.join(output_dir, 'tsne_projection.png'), os.path.basename(os.path.join(output_dir, 'tsne_projection.png')))
print(f"All results compressed into {zip_filename}")

# 3. Google Drive API를 사용하여 ZIP 파일 업로드
# Colab의 내장 인증 방식을 사용합니다.
drive_service = build('drive', 'v3')

if os.path.exists(zip_filename):
    file_metadata = {
        'name': os.path.basename(zip_filename),
        'parents': [drive_folder_id]
    }
    media = MediaFileUpload(zip_filename, resumable=True)
    uploaded_file = drive_service.files().create(
        body=file_metadata,
        media_body=media,
        fields='id'
    ).execute()
    print(f"Uploaded '{os.path.basename(zip_filename)}' to Google Drive folder ID: {drive_folder_id}. File ID: {uploaded_file.get('id')}")
else:
    print(f"Warning: ZIP file not found for upload - {zip_filename}")

print("All specified files have been processed and uploaded to Google Drive as a ZIP archive.")